# NVIDIA Libraries Tour

**Before running:** `Runtime → Change runtime type → GPU` (T4 minimum)

Libraries covered:
| Library | What it is |
|---------|------------|
| **CuPy** | GPU NumPy. Gateway to cuBLAS, cuFFT, cuRAND, cuSPARSE, cuSolver |
| **cuDF** | GPU pandas. cuIO built in (CSV, Parquet, JSON) |
| **cuML** | GPU scikit-learn. 50+ algorithms |
| **cuGraph** | GPU NetworkX. Graph algorithms at scale |

## Cell 1 — Install RAPIDS (run once, takes ~3 min)

In [ ]:
import subprocess, sys

# Official RAPIDS install script for Colab — detects GPU + CUDA version automatically
subprocess.run(['git','clone','--depth=1',
    'https://github.com/rapidsai/rapidsai-csp-utils.git'], check=True)
subprocess.run([sys.executable,
    'rapidsai-csp-utils/colab/pip-install.py'], check=True)
print('RAPIDS installed.')

In [ ]:
# Verify GPU and all libraries
import cupy as cp
import cudf, cuml, cugraph

gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
free_gb, total_gb = [x/1e9 for x in cp.cuda.runtime.memGetInfo()]
print(f'GPU      : {gpu_name}')
print(f'VRAM     : {total_gb:.1f} GB total  {free_gb:.1f} GB free')
print(f'CuPy     : {cp.__version__}')
print(f'cuDF     : {cudf.__version__}')
print(f'cuML     : {cuml.__version__}')
print(f'cuGraph  : {cugraph.__version__}')

---
## Part 1 — CuPy
**CuPy = NumPy on GPU.** Every call dispatches to a low-level NVIDIA library.

```
cp.matmul   →  cuBLAS  (matrix multiply)
cp.fft.*    →  cuFFT   (Fourier transform)
cp.random.* →  cuRAND  (random numbers)
cupyx.scipy.sparse → cuSPARSE
cp.linalg.* →  cuSolver
cp.RawKernel → write your own CUDA C kernel
```

### 1a. GPU Arrays & cuBLAS (matrix multiply)

In [ ]:
import time, numpy as np
import cupy as cp

N = 4096
A = cp.random.randn(N, N, dtype=cp.float32)
B = cp.random.randn(N, N, dtype=cp.float32)

# Warm-up
_ = cp.matmul(A, B)
cp.cuda.stream.get_current_stream().synchronize()

t0 = time.perf_counter()
C = cp.matmul(A, B)   # calls cuBLAS sgemm
cp.cuda.stream.get_current_stream().synchronize()
gpu_ms = (time.perf_counter() - t0) * 1000

A_cpu, B_cpu = cp.asnumpy(A), cp.asnumpy(B)
t0 = time.perf_counter()
_ = np.matmul(A_cpu, B_cpu)
cpu_ms = (time.perf_counter() - t0) * 1000

print(f'cuBLAS GPU : {gpu_ms:.1f} ms')
print(f'NumPy  CPU : {cpu_ms:.1f} ms')
print(f'Speedup    : {cpu_ms/gpu_ms:.1f}x')

### 1b. cuFFT — Fast Fourier Transform

In [ ]:
N = 1 << 20
t = cp.linspace(0, 1, N, dtype=cp.float32)
signal = cp.sin(2*cp.pi*50*t) + 0.5*cp.sin(2*cp.pi*120*t)

t0 = time.perf_counter()
spectrum = cp.fft.rfft(signal)   # calls cuFFT
cp.cuda.stream.get_current_stream().synchronize()
fft_ms = (time.perf_counter() - t0) * 1000

freqs = cp.fft.rfftfreq(N, d=1.0/N)
peak_freq = float(freqs[int(cp.argmax(cp.abs(spectrum)))])
print(f'cuFFT on {N:,} samples : {fft_ms:.2f} ms')
print(f'Peak frequency detected: {peak_freq:.0f} Hz  (expected 50)')

### 1c. cuRAND — Random Number Generation

In [ ]:
N = 10_000_000
uniform  = cp.random.rand(N, dtype=cp.float32)       # Uniform [0,1) via cuRAND
normal   = cp.random.randn(N).astype(cp.float32)     # Normal  via cuRAND
poisson  = cp.random.poisson(lam=5.0, size=N)        # Poisson via cuRAND

print(f'Uniform  mean: {float(uniform.mean()):.4f}  (want 0.5)')
print(f'Normal   mean: {float(normal.mean()):.4f}  std: {float(normal.std()):.4f}')
print(f'Poisson  mean: {float(poisson.mean()):.4f}  (want 5.0)')

### 1d. cuSPARSE — Sparse Matrix Operations

In [ ]:
import cupyx.scipy.sparse as sp

n = 5000
rng = np.random.default_rng(0)
row  = rng.integers(0, n, 25000).astype(np.int32)
col  = rng.integers(0, n, 25000).astype(np.int32)
data = rng.standard_normal(25000).astype(np.float32)

A_sp = sp.csr_matrix(
    (cp.asarray(data), (cp.asarray(row), cp.asarray(col))),
    shape=(n, n)
)
x    = cp.random.randn(n, dtype=cp.float32)

t0   = time.perf_counter()
y    = A_sp @ x    # cuSPARSE SpMV
cp.cuda.stream.get_current_stream().synchronize()
print(f'cuSPARSE SpMV {n}x{n} (nnz={A_sp.nnz:,}): {(time.perf_counter()-t0)*1000:.2f} ms')
print(f'Result norm: {float(cp.linalg.norm(y)):.4f}')

### 1e. Custom CUDA Kernel with cp.RawKernel

In [ ]:
cuda_code = r"""
extern "C" __global__
void scaled_add(const float* a, const float* b, float* c, float alpha, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = alpha * a[i] + b[i];
}
"""
kernel = cp.RawKernel(cuda_code, 'scaled_add')

N = 1_000_000
a_gpu = cp.random.randn(N, dtype=cp.float32)
b_gpu = cp.random.randn(N, dtype=cp.float32)
c_gpu = cp.empty(N, dtype=cp.float32)
alpha = cp.float32(2.5)

threads = 256
blocks  = (N + threads - 1) // threads
kernel((blocks,), (threads,), (a_gpu, b_gpu, c_gpu, alpha, N))
cp.cuda.stream.get_current_stream().synchronize()

expected = float(alpha) * a_gpu + b_gpu
print(f'Custom kernel correct: {cp.allclose(c_gpu, expected)}')
print(f'Max error           : {float(cp.max(cp.abs(c_gpu - expected))):.2e}')

### 1f. CUDA Streams — async parallel execution

In [ ]:
s1 = cp.cuda.Stream()
s2 = cp.cuda.Stream()
X1 = cp.random.randn(2000, 2000, dtype=cp.float32)
X2 = cp.random.randn(2000, 2000, dtype=cp.float32)

with s1: R1 = cp.matmul(X1, X1.T)   # runs on stream 1
with s2: R2 = cp.fft.fft2(X2)       # runs on stream 2 at the SAME TIME

s1.synchronize(); s2.synchronize()
print(f'Stream1 matmul shape : {R1.shape}')
print(f'Stream2 fft2   shape : {R2.shape}')
print('Both ran concurrently on the GPU.')

---
## Part 2 — cuDF + cuIO
**cuDF = pandas on GPU. cuIO = I/O that writes directly to GPU memory.**
Data never touches the CPU during a pipeline — no PCIe transfer overhead.

In [ ]:
import cudf, io as pyio

# ── DataFrame from dict ──
df = cudf.DataFrame({
    'name'  : ['Alice','Bob','Carol','Dave','Eve'],
    'age'   : [25, 30, 35, 28, 22],
    'salary': [70000, 90000, 85000, 60000, 55000],
    'dept'  : ['Eng','Eng','HR','Sales','Eng'],
})
print('DataFrame from dict:')
print(df)

In [ ]:
# ── cuIO: read CSV directly to GPU ──
csv_text = 'id,product,price\n1,Widget,9.99\n2,Gadget,24.99\n3,Doohickey,4.49\n'
csv_df = cudf.read_csv(pyio.StringIO(csv_text))
print('CSV loaded directly to GPU:')
print(csv_df)

# Large DataFrame
rng2 = np.random.default_rng(1)
N = 1_000_000
big = cudf.DataFrame({
    'x'    : rng2.standard_normal(N).astype(np.float32),
    'y'    : rng2.standard_normal(N).astype(np.float32),
    'label': rng2.integers(0, 5, N).astype(np.int32),
})
print(f'Large DataFrame: {len(big):,} rows')

In [ ]:
# ── cuIO: Parquet write + read ──
big.to_parquet('/tmp/big.parquet', index=False)
t0 = time.perf_counter()
loaded = cudf.read_parquet('/tmp/big.parquet')
print(f'Parquet read {len(loaded):,} rows in {(time.perf_counter()-t0)*1000:.1f} ms')

# ── GroupBy ──
t0 = time.perf_counter()
agg = big.groupby('label').agg({'x': ['mean','std'], 'y': 'mean'})
print(f'GroupBy on {len(big):,} rows : {(time.perf_counter()-t0)*1000:.1f} ms')
print(agg)

In [ ]:
# ── Merge / Join ──
orders = cudf.DataFrame({'order_id':[101,102,103], 'product_id':[1,3,2], 'qty':[2,1,5]})
products = cudf.DataFrame({'product_id':[1,2,3], 'name':['Widget','Gadget','Doohickey'],
                            'price':[9.99,24.99,4.49]})
joined = orders.merge(products, on='product_id', how='inner')
joined['total'] = joined['qty'] * joined['price']
print('Joined orders:')
print(joined[['order_id','name','qty','total']])

# ── String ops ──
emails = cudf.Series(['Alice@Example.COM', 'bob@company.org', '  carol@test.net  '])
clean = emails.str.strip().str.lower()
domains = clean.str.extract(r'@([\w.]+)')
print('\nExtracted domains:')
print(domains)

---
## Part 3 — cuML
**cuML = scikit-learn on GPU.** Same `.fit()` / `.predict()` API, works with cuDF DataFrames.

In [ ]:
from cuml.linear_model   import LinearRegression, LogisticRegression, Ridge
from cuml.ensemble       import RandomForestClassifier
from cuml.cluster        import KMeans, DBSCAN
from cuml.decomposition  import PCA
from cuml.manifold       import UMAP
from cuml.preprocessing  import StandardScaler
from cuml.model_selection import train_test_split
from cuml.metrics        import accuracy_score, r2_score

# ── Synthetic dataset ──
rng3 = np.random.default_rng(0)
N3, D = 100_000, 20
true_w = rng3.standard_normal(D).astype(np.float32)
X_np   = rng3.standard_normal((N3, D)).astype(np.float32)
y_reg  = X_np @ true_w + rng3.standard_normal(N3).astype(np.float32) * 0.1
y_cls  = (X_np[:,0] > 0).astype(np.int32)

X_gdf  = cudf.DataFrame(X_np)
yr_ser = cudf.Series(y_reg)
yc_ser = cudf.Series(y_cls)

Xtr,Xte,ytr,yte = train_test_split(X_gdf, yr_ser, test_size=0.2, random_state=0)
_,  __,ctr,cte  = train_test_split(X_gdf, yc_ser, test_size=0.2, random_state=0)
print(f'Train: {len(Xtr):,}   Test: {len(Xte):,}')

In [ ]:
# ── Linear Regression (cuBLAS backend) ──
t0 = time.perf_counter()
lr = LinearRegression(fit_intercept=True)
lr.fit(Xtr, ytr)
print(f'LinearRegression fit  : {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'R²                    : {float(r2_score(yte, lr.predict(Xte))):.4f}')

# ── Logistic Regression ──
t0 = time.perf_counter()
log = LogisticRegression(max_iter=500, C=1.0)
log.fit(Xtr, ctr)
print(f'\nLogisticRegression fit: {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Accuracy              : {float(accuracy_score(cte, log.predict(Xte))):.4f}')

In [ ]:
# ── Random Forest ──
t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0)
rf.fit(Xtr, ctr)
print(f'RandomForest fit      : {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Accuracy              : {float(accuracy_score(cte, rf.predict(Xte))):.4f}')

# ── KMeans ──
centers = rng3.standard_normal((5, D)).astype(np.float32) * 5
Xk = np.vstack([rng3.standard_normal((2000, D)).astype(np.float32)+centers[i] for i in range(5)])
Xk_gdf = cudf.DataFrame(Xk)

t0 = time.perf_counter()
km = KMeans(n_clusters=5, random_state=0)
km.fit(Xk_gdf)
print(f'\nKMeans fit (10k pts)  : {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Inertia               : {float(km.inertia_):.2f}')

In [ ]:
# ── PCA ──
pca = PCA(n_components=5, random_state=0)
t0 = time.perf_counter()
pca.fit(X_gdf)
Xr = pca.transform(X_gdf)
print(f'PCA fit+transform ({N3:,} × {D} → {D} → 5): {(time.perf_counter()-t0)*1000:.1f} ms')
explained = float(cp.asarray(pca.explained_variance_ratio_).sum()) * 100
print(f'Variance explained by 5 PCs: {explained:.1f}%')

# ── UMAP ──
umap_model = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=0)
t0 = time.perf_counter()
emb = umap_model.fit_transform(cudf.DataFrame(Xk))
print(f'\nUMAP fit_transform (10k pts → 2D): {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Embedding shape: {emb.shape}')

---
## Part 4 — cuGraph
**cuGraph = NetworkX on GPU.** Reads cuDF edge-list DataFrames natively.

In [ ]:
import cugraph

rng4 = np.random.default_rng(42)
N_nodes, N_edges = 50_000, 500_000
edges = cudf.DataFrame({
    'src': cudf.Series(rng4.integers(0, N_nodes, N_edges).astype(np.int32)),
    'dst': cudf.Series(rng4.integers(0, N_nodes, N_edges).astype(np.int32)),
    'wgt': cudf.Series(rng4.uniform(0.1, 10, N_edges).astype(np.float32)),
})

G = cugraph.Graph()
t0 = time.perf_counter()
G.from_cudf_edgelist(edges, source='src', destination='dst', edge_attr='wgt')
print(f'Build graph ({N_nodes:,} nodes, {N_edges:,} edges): {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Nodes: {G.number_of_nodes():,}   Edges: {G.number_of_edges():,}')

In [ ]:
# ── PageRank ──
t0 = time.perf_counter()
pr = cugraph.pagerank(G, alpha=0.85, max_iter=100)
print(f'PageRank : {(time.perf_counter()-t0)*1000:.1f} ms')
print('Top-5 nodes:')
print(pr.sort_values('pagerank', ascending=False).head())

# ── BFS ──
t0 = time.perf_counter()
bfs = cugraph.bfs(G, start=0)
print(f'\nBFS from node 0  : {(time.perf_counter()-t0)*1000:.1f} ms')
reachable = int((bfs['distance'] < 2**31-1).sum())
print(f'Reachable nodes  : {reachable:,}')

In [ ]:
# ── SSSP ──
t0 = time.perf_counter()
sssp = cugraph.sssp(G, source=0)
print(f'SSSP from node 0 : {(time.perf_counter()-t0)*1000:.1f} ms')
finite = sssp[sssp['distance'] < 1e38]
print(f'Avg weighted dist: {float(finite["distance"].mean()):.4f}')

# ── Louvain Community Detection ──
t0 = time.perf_counter()
parts, modularity = cugraph.louvain(G, max_iter=100)
print(f'\nLouvain          : {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Communities found: {len(parts["partition"].unique()):,}')
print(f'Modularity       : {modularity:.4f}')

# ── Triangle Count ──
t0 = time.perf_counter()
tc = cugraph.triangle_count(G)
print(f'\nTriangle Count   : {(time.perf_counter()-t0)*1000:.1f} ms')
print(f'Total triangles  : {int(tc["counts"].sum())//3:,}')

# ── Weakly Connected Components ──
wcc = cugraph.weakly_connected_components(G)
print(f'\nConnected components: {len(wcc["labels"].unique()):,}')

---
## Part 5 — Full RAPIDS Pipeline
**cuDF → cuML → cuGraph all on GPU, zero CPU transfers.**

In [ ]:
print('=== FULL RAPIDS PIPELINE ===')
t_pipeline = time.perf_counter()

# Step 1: Generate + load data with cuDF
rng5 = np.random.default_rng(7)
N5 = 200_000
raw = cudf.DataFrame({
    'user_id'  : rng5.integers(0, 10_000, N5).astype(np.int32),
    'item_id'  : rng5.integers(0, 5_000, N5).astype(np.int32),
    'rating'   : rng5.uniform(1, 5, N5).astype(np.float32),
    'feat_a'   : rng5.standard_normal(N5).astype(np.float32),
    'feat_b'   : rng5.standard_normal(N5).astype(np.float32),
    'feat_c'   : rng5.standard_normal(N5).astype(np.float32),
})
print(f'Step 1 — Raw data: {len(raw):,} rows')

# Step 2: cuDF preprocessing — normalise + aggregate
raw['label'] = (raw['rating'] >= 4.0).astype(np.int32)
user_stats = raw.groupby('user_id').agg(
    {'rating':'mean', 'feat_a':'mean', 'feat_b':'mean'}
).reset_index()
user_stats.columns = cudf.Index(['user_id','avg_rating','mean_a','mean_b'])
print(f'Step 2 — User aggregates: {len(user_stats):,} users')

# Step 3: cuML — cluster users
feat_cols = cudf.DataFrame({'mean_a': user_stats['mean_a'],
                             'mean_b': user_stats['mean_b'],
                             'avg_rating': user_stats['avg_rating']})
sc = StandardScaler()
feat_scaled = sc.fit_transform(feat_cols)
km = KMeans(n_clusters=5, random_state=0)
km.fit(feat_scaled)
user_stats['cluster'] = km.labels_
print(f'Step 3 — KMeans clusters: {len(user_stats["cluster"].unique())}')

# Step 4: cuML — predict high-rating probability
features = raw[['feat_a','feat_b','feat_c']]
labels   = raw['label']
Xtr5, Xte5, ytr5, yte5 = train_test_split(features, labels, test_size=0.2, random_state=0)
clf5 = LogisticRegression(max_iter=500)
clf5.fit(Xtr5, ytr5)
acc5 = float(accuracy_score(yte5, clf5.predict(Xte5)))
print(f'Step 4 — Rating classifier accuracy: {acc5:.4f}')

# Step 5: cuGraph — build user-item graph, find top users by PageRank
edge_sample = raw[['user_id','item_id','rating']].head(50_000)
G5 = cugraph.Graph()
G5.from_cudf_edgelist(edge_sample, source='user_id',
                       destination='item_id', edge_attr='rating')
pr5 = cugraph.pagerank(G5)
top_users5 = pr5[pr5['vertex'] < 10_000].sort_values(
    'pagerank', ascending=False
).head()
print(f'Step 5 — Top users by PageRank:\n{top_users5}')

total_ms = (time.perf_counter() - t_pipeline) * 1000
print(f'\n=== Total pipeline time: {total_ms:.0f} ms ({N5:,} rows, all on GPU) ===')

---
## Library Reference Card

| What you want | CuPy call | NVIDIA lib |
|---------------|-----------|------------|
| Matrix multiply | `cp.matmul(A, B)` | cuBLAS |
| FFT | `cp.fft.fft(x)` | cuFFT |
| Random normal | `cp.random.randn(n)` | cuRAND |
| Sparse matvec | `A_csr @ x` | cuSPARSE |
| Linear solve | `cp.linalg.solve(A, b)` | cuSolver |
| SVD | `cp.linalg.svd(A)` | cuSolver |
| Custom kernel | `cp.RawKernel(code, name)` | NVRTC + CUDA |
| GPU DataFrame | `cudf.DataFrame(...)` | cuDF |
| Read CSV to GPU | `cudf.read_csv(path)` | cuIO |
| GPU ML model | `cuml.linear_model.LinearRegression()` | cuML |
| Graph analytics | `cugraph.pagerank(G)` | cuGraph |
